In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.spark_session import get_spark_session, load_config
from src.loader import load_yellow_taxi_data

spark = get_spark_session("FeaturesExploration")
config = load_config()

df = load_yellow_taxi_data(
    spark,
    "data/processed/yellow_taxi_features",
    apply_schema=False
)

print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")

your 131072x1 screen size is bogus. expect trouble
26/06/12 17:47:43 WARN Utils: Your hostname, DESKTOP-5HFQKG4 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/12 17:47:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 17:47:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Rows: 2,704,152
Columns: 38


In [2]:
# Sample with new columns
df.select(
    "pickup_hour",
    "pickup_day_name",
    "is_weekend",
    "time_of_day",
    "is_rush_hour",
    "speed_mph",
    "tip_percentage",
    "trip_length_category",
    "payment_type_name"
).show(10, truncate=False)

+-----------+---------------+----------+-----------+------------+---------+--------------+--------------------+-----------------+
|pickup_hour|pickup_day_name|is_weekend|time_of_day|is_rush_hour|speed_mph|tip_percentage|trip_length_category|payment_type_name|
+-----------+---------------+----------+-----------+------------+---------+--------------+--------------------+-----------------+
|0          |Wednesday      |false     |Night      |false       |17.62    |26.13         |Medium              |Credit card      |
|0          |Wednesday      |false     |Night      |false       |14.04    |42.15         |Medium              |Credit card      |
|0          |Wednesday      |false     |Night      |false       |30.85    |30.92         |Long                |Credit card      |
|0          |Wednesday      |false     |Night      |false       |9.75     |30.9          |Medium              |Credit card      |
|0          |Wednesday      |false     |Night      |false       |6.12     |0.0           |

In [3]:
from pyspark.sql.functions import count, avg, round as spark_round

df.groupBy("pickup_hour") \
  .agg(
      count("*").alias("trips"),
      spark_round(avg("fare_amount"), 2).alias("avg_fare"),
      spark_round(avg("speed_mph"), 1).alias("avg_speed_mph")
  ) \
  .orderBy("pickup_hour") \
  .show(24)

+-----------+------+--------+-------------+
|pickup_hour| trips|avg_fare|avg_speed_mph|
+-----------+------+--------+-------------+
|          0|124975|   19.17|         13.7|
|          1| 93953|   20.34|         14.7|
|          2| 68766|   19.65|         14.7|
|          3| 45051|   17.31|         13.9|
|          4| 31699|   16.32|         13.8|
|          5| 20356|   17.85|         15.2|
|          6| 12636|   22.94|         18.0|
|          7| 15474|   27.54|         19.6|
|          8| 35398|   21.93|         15.9|
|          9| 74008|   18.49|         12.9|
|         10|104741|   17.55|         11.0|
|         11|118926|   17.82|         11.0|
|         12|129548|   17.99|         10.8|
|         13|140717|   17.56|         10.3|
|         14|153026|   17.73|         10.3|
|         15|158418|   18.34|         10.4|
|         16|170501|   19.18|         10.2|
|         17|176184|   19.02|          9.9|
|         18|177610|   19.37|         10.2|
|         19|190494|   18.02|   

In [4]:
df.groupBy("is_weekend") \
  .agg(
      count("*").alias("trips"),
      spark_round(avg("trip_distance"), 2).alias("avg_distance"),
      spark_round(avg("fare_amount"), 2).alias("avg_fare"),
      spark_round(avg("tip_percentage"), 2).alias("avg_tip_pct")
  ) \
  .show()

+----------+-------+------------+--------+-----------+
|is_weekend|  trips|avg_distance|avg_fare|avg_tip_pct|
+----------+-------+------------+--------+-----------+
|      true| 710414|        3.19|   17.63|      21.63|
|     false|1993738|        3.33|   18.66|      21.37|
+----------+-------+------------+--------+-----------+



In [5]:
df.groupBy("payment_type_name") \
  .agg(
      count("*").alias("trips"),
      spark_round(avg("tip_percentage"), 2).alias("avg_tip_pct"),
      spark_round(avg("total_amount"), 2).alias("avg_total")
  ) \
  .orderBy("trips", ascending=False) \
  .show()

+-----------------+-------+-----------+---------+
|payment_type_name|  trips|avg_tip_pct|avg_total|
+-----------------+-------+-----------+---------+
|      Credit card|2261235|      25.64|    28.04|
|             Cash| 412890|        0.0|    23.83|
|          Dispute|  21116|       0.03|    25.32|
|        No charge|   8911|       0.04|    22.66|
+-----------------+-------+-----------+---------+



In [6]:
df.groupBy("is_airport_trip") \
  .agg(
      count("*").alias("trips"),
      spark_round(avg("trip_distance"), 2).alias("avg_distance"),
      spark_round(avg("fare_amount"), 2).alias("avg_fare"),
      spark_round(avg("total_amount"), 2).alias("avg_total"),
      spark_round(avg("tip_percentage"), 2).alias("avg_tip_pct")
  ) \
  .show()

+---------------+-------+------------+--------+---------+-----------+
|is_airport_trip|  trips|avg_distance|avg_fare|avg_total|avg_tip_pct|
+---------------+-------+------------+--------+---------+-----------+
|           true| 271546|       13.67|   56.02|    77.25|      17.35|
|          false|2432606|        2.14|   14.19|    21.79|       21.9|
+---------------+-------+------------+--------+---------+-----------+



In [7]:
spark.stop()